# **Validation of ML model from Github Repo**

In [0]:
import os
repo_path = os.path.abspath("..")
print(repo_path)
print(os.listdir(repo_path))

Install libraries requeriments

In [0]:
%pip install -r ../models/lgbm_sm/requirements.txt

Restart Notebook kernel

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import pandas as pd
import numpy as np
import lightgbm

print("MLflow Version: %s" % mlflow.__version__)
print("Pandas Version: %s" % pd.__version__)
print("Numpy Version: %s" % np.__version__)
print("LightGBM Version: %s" % lightgbm.__version__)


## LETS LOAD THE MODEL

In [0]:
import os
import mlflow

repo_path = os.path.abspath("..")
model_path = os.path.join(repo_path, "models", "lgbm_sm")

model = mlflow.pyfunc.load_model(model_path)
print("Model loaded successfully")



Exploring the model in order to know which inputs features needs the model.

In [0]:
from mlflow.models import Model

mlmodel = Model.load(model_path)
print(mlmodel.signature)

## Testing the model with autogenerated data

In [0]:
import pandas as pd
import numpy as np

input_data = pd.DataFrame([
    {
        "CreditScore": 650,
        "Geography_France": False,
        "Geography_Germany": False,
        "Geography_Spain": True,
        "Gender_Female": False,
        "Gender_Male": True,
        "Age": 35,
        "Tenure": 5,
        "Balance": 60000.0,
        "NumOfProducts": 2,
        "HasCrCard": 1,
        "IsActiveMember": 1,
        "EstimatedSalary": 50000.0,
        "NewTenure": 5 / 35,
        "NewCreditsScore": 4,
        "NewAgeScore": 3,
        "NewBalanceScore": 3,
        "NewEstSalaryScore": 5
    },
    {
        "CreditScore": 720,
        "Geography_France": False,
        "Geography_Germany": True,
        "Geography_Spain": False,
        "Gender_Female": True,
        "Gender_Male": False,
        "Age": 48,
        "Tenure": 2,
        "Balance": 130000.0,
        "NumOfProducts": 1,
        "HasCrCard": 1,
        "IsActiveMember": 0,
        "EstimatedSalary": 90000.0,
        "NewTenure": 2 / 48,
        "NewCreditsScore": 5,
        "NewAgeScore": 6,
        "NewBalanceScore": 5,
        "NewEstSalaryScore": 8
    },
    {
        "CreditScore": 590,
        "Geography_France": True,
        "Geography_Germany": False,
        "Geography_Spain": False,
        "Gender_Female": False,
        "Gender_Male": True,
        "Age": 28,
        "Tenure": 8,
        "Balance": 0.0,
        "NumOfProducts": 2,
        "HasCrCard": 0,
        "IsActiveMember": 1,
        "EstimatedSalary": 30000.0,
        "NewTenure": 8 / 28,
        "NewCreditsScore": 2,
        "NewAgeScore": 2,
        "NewBalanceScore": 1,
        "NewEstSalaryScore": 3
    }
])

input_data = input_data.astype({
    "CreditScore": np.int32,
    "Geography_France": bool,
    "Geography_Germany": bool,
    "Geography_Spain": bool,
    "Gender_Female": bool,
    "Gender_Male": bool,
    "Age": np.int32,
    "Tenure": np.int64,
    "Balance": np.float64,
    "NumOfProducts": np.int32,
    "HasCrCard": np.int32,
    "IsActiveMember": np.int32,
    "EstimatedSalary": np.float64,
    "NewTenure": np.float64,
    "NewCreditsScore": np.int64,
    "NewAgeScore": np.int64,
    "NewBalanceScore": np.int64,
    "NewEstSalaryScore": np.int64
})

print(input_data.dtypes)

predictions = model.predict(input_data)

results = input_data.copy()
results["prediction"] = predictions

output_dir = "/tmp/bank_churn/predictions"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'output.csv')
results.to_csv(output_path, index=False)

print("Output path: ", output_path)
print("File exists:", os.path.exists(output_path))
display(results)

## Track Validation with MLflow Experiments

In [0]:
import mlflow

mlflow.set_experiment("/Users/admin@graxiano.com/bank-churn-fabric/validate_fabric_model_in_databricks")

with mlflow.start_run(run_name = "fabric_model_validation_in_databricks") as run:

    mlflow.log_param("source_platform", "Microsoft Fabric")
    mlflow.log_param("model_format", "Mlflow artifacrt")
    mlflow.log_param("repository", "Github")
    mlflow.log_param("target_platform", "Databricks")
    mlflow.log_param("model_name", "lgbm_sm")
    mlflow.log_param("validation_type", "cross_platform_reproducibility")

    mlflow.log_metric("num_input_records", len(input_data))
    mlflow.log_metric("num_predictions", len(predictions))
    mlflow.log_metric("num_predict_churn", int(sum(predictions)))

    mlflow.log_artifact(output_path)

    run_id = run.info.run_id

print("Validation run logged succesfully")
print("Run ID:", run_id)

## REGISTER MODEL IN DATABRICKS MODELS

In [0]:
print('ll')

In [0]:
import mlflow

registered_model = mlflow.register_model(
    model_uri = model_path,
    name = "bank_sm_churn_model"
)

print("Model registered successfully")
print("Name:", registered_model.name)
print("Version:", registered_model.version)



## LOAD MODEL FROM REGISTER MODELS

In [0]:
model_registered = mlflow.pyfunc.load_model("models:/bank_sm_churn_model/1")

predictions_registry = model_registered.predict(input_data)

print(predictions_registry)